In [1]:

import pandas as pd
import numpy as np
import joblib
import json

from pathlib import Path

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

from sklearn.ensemble import (
    RandomForestRegressor,
    RandomForestClassifier
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    precision_recall_curve,
    r2_score
)

print("All libraries imported successfully!")


All libraries imported successfully!


In [2]:

# CELL 2: Load dataset safely

CSV_CANDIDATES = [
    Path("pune_multi_disease_weekly_synthetic_7years(1)(2).csv"),
    Path("pune_multi_disease_weekly_synthetic_7years(1).csv"),
    Path("pune_multi_disease_weekly_synthetic_7years.csv"),
]

CSV_PATH = next(
    (path for path in CSV_CANDIDATES if path.exists()),
    None
)

# If the listed names do not match, try other versions.
if CSV_PATH is None:
    matches = sorted(
        Path(".").glob(
            "pune_multi_disease_weekly_synthetic_7years*.csv"
        )
    )

    if matches:
        CSV_PATH = matches[0]

if CSV_PATH is None:
    raise FileNotFoundError(
        "Dataset nahi mili. CSV ko notebook ke folder mein rakho."
    )

df = pd.read_csv(CSV_PATH)

print("Dataset loaded successfully!")
print("File:", CSV_PATH.resolve())
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

display(df.head())


Dataset loaded successfully!
File: C:\Users\YASH\Prime Classes\health2\pune_multi_disease_weekly_synthetic_7years(1).csv
Rows: 7300
Columns: 79


,date,city,location,latitude,longitude,week_of_year,year,temperature_avg_c,humidity_avg_pct,rainfall_mm,...,target_typhoid_hospitalizations_next_week,target_typhoid_deaths_next_week,acute_diarrheal_disease_cases_lag_1w,acute_diarrheal_disease_cases_lag_2w,acute_diarrheal_disease_cases_lag_4w,acute_diarrheal_disease_cases_rolling_mean_4w,target_acute_diarrheal_disease_cases_next_week,acute_diarrheal_disease_spike_next_week,target_acute_diarrheal_disease_hospitalizations_next_week,target_acute_diarrheal_disease_deaths_next_week
0,2019-01-07,Pune,Aundh,18.558,73.807,2,2019,23.5,61.1,0.0,...,0.0,1.0,NaN,NaN,NaN,NaN,5.0,NaN,1.0,0.0
1,2019-01-14,Pune,Aundh,18.558,73.807,3,2019,24.6,65.8,29.8,...,1.0,0.0,5.0,NaN,NaN,NaN,6.0,NaN,0.0,0.0
2,2019-01-21,Pune,Aundh,18.558,73.807,4,2019,24.2,67.3,23.1,...,0.0,0.0,5.0,5.0,NaN,NaN,7.0,NaN,0.0,1.0
3,2019-01-28,Pune,Aundh,18.558,73.807,5,2019,24.1,49.9,7.8,...,0.0,0.0,6.0,5.0,NaN,NaN,9.0,NaN,0.0,0.0
4,2019-02-04,Pune,Aundh,18.558,73.807,6,2019,23.8,65.0,13.0,...,1.0,0.0,7.0,6.0,5.0,5.75,0.0,0.0,0.0,0.0


In [3]:
####
# 1. Check column names and data types
print("Dataset shape:", df.shape)

print("\nColumn names and data types:")
display(
    pd.DataFrame({
        "Column": df.columns,
        "Data Type": df.dtypes.astype(str).values,
        "Missing Values": df.isnull().sum().values,
        "Missing %": (
            df.isnull().mean().values * 100
        ).round(2)
    })
)

# 2. Check duplicate rows
print("\nDuplicate rows:", df.duplicated().sum())

# 3. Check date range
df["date"] = pd.to_datetime(df["date"], errors="coerce")

print("\nStart date:", df["date"].min())
print("End date:", df["date"].max())
print("Unique locations:", df["location"].nunique())
print("Unique dates:", df["date"].nunique())

# 4. Check disease-related columns
print("\nAll disease-related columns:")
for col in df.columns:
    if any(
        disease in col
        for disease in [
            "dengue",
            "malaria",
            "chikungunya",
            "influenza",
            "typhoid",
            "acute_diarrheal_disease"
        ]
    ):
        print(col)

Dataset shape: (7300, 79)

Column names and data types:


,Column,Data Type,Missing Values,Missing %
0,date,object,0,0.00
1,city,object,0,0.00
2,location,object,0,0.00
3,latitude,float64,0,0.00
4,longitude,float64,0,0.00
...,...,...,...,...
74,acute_diarrheal_disease_cases_rolling_mean_4w,float64,80,1.10
75,target_acute_diarrheal_disease_cases_next_week,float64,20,0.27
76,acute_diarrheal_disease_spike_next_week,float64,100,1.37
77,target_acute_diarrheal_disease_hospitalization...,float64,20,0.27



Duplicate rows: 0

Start date: 2019-01-07 00:00:00
End date: 2025-12-29 00:00:00
Unique locations: 20
Unique dates: 365

All disease-related columns:
dengue_cases
malaria_cases
chikungunya_cases
influenza_cases
typhoid_cases
acute_diarrheal_disease_cases
dengue_hospitalizations
dengue_deaths
malaria_hospitalizations
malaria_deaths
chikungunya_hospitalizations
chikungunya_deaths
influenza_hospitalizations
influenza_deaths
typhoid_hospitalizations
typhoid_deaths
acute_diarrheal_disease_hospitalizations
acute_diarrheal_disease_deaths
dengue_cases_lag_1w
dengue_cases_lag_2w
dengue_cases_lag_4w
dengue_cases_rolling_mean_4w
target_dengue_cases_next_week
dengue_spike_next_week
target_dengue_hospitalizations_next_week
target_dengue_deaths_next_week
malaria_cases_lag_1w
malaria_cases_lag_2w
malaria_cases_lag_4w
malaria_cases_rolling_mean_4w
target_malaria_cases_next_week
malaria_spike_next_week
target_malaria_hospitalizations_next_week
target_malaria_deaths_next_week
chikungunya_cases_lag_1w
c

In [4]:
print("Date data type:", df["date"].dtype)
print("Duplicate rows:", df.duplicated().sum())
print("Start date:", df["date"].min())
print("End date:", df["date"].max())
print("Unique locations:", df["location"].nunique())
print("Unique dates:", df["date"].nunique())

Date data type: datetime64[ns]
Duplicate rows: 0
Start date: 2019-01-07 00:00:00
End date: 2025-12-29 00:00:00
Unique locations: 20
Unique dates: 365


In [5]:

diseases = [
    "dengue",
    "malaria",
    "chikungunya",
    "influenza",
    "typhoid",
    "acute_diarrheal_disease"
]

# 1. Check missing values in important target columns
target_cols = [
    col for col in df.columns
    if col.startswith("target_")
    or col.endswith("_spike_next_week")
]

print("Number of target columns:", len(target_cols))
print("\nMissing values in targets:")

display(
    df[target_cols]
    .isnull()
    .sum()
    .sort_values(ascending=False)
    .to_frame("Missing_Count")
)

# 2. Check how many valid targets each disease has
print("\nValid next-week case targets:")

for disease in diseases:
    target = f"target_{disease}_cases_next_week"
    spike = f"{disease}_spike_next_week"

    print(
        f"{disease}: "
        f"case targets = {df[target].notna().sum()}, "
        f"spike labels = {df[spike].notna().sum()}"
    )

# 3. Check missing rows at the end of the timeline
print("\nLatest dates in dataset:")
display(
    df.sort_values("date")
      .groupby("date")
      .size()
      .tail(5)
      .to_frame("Rows_per_date")
)

# 4. Check spike label values
print("\nSpike label values:")
for disease in diseases:
    col = f"{disease}_spike_next_week"
    print(disease, df[col].value_counts(dropna=False).to_dict())

Number of target columns: 24

Missing values in targets:


,Missing_Count
acute_diarrheal_disease_spike_next_week,100
malaria_spike_next_week,100
typhoid_spike_next_week,100
chikungunya_spike_next_week,100
dengue_spike_next_week,100
influenza_spike_next_week,100
target_dengue_cases_next_week,20
target_influenza_hospitalizations_next_week,20
target_acute_diarrheal_disease_hospitalizations_next_week,20
target_acute_diarrheal_disease_cases_next_week,20



Valid next-week case targets:
dengue: case targets = 7280, spike labels = 7200
malaria: case targets = 7280, spike labels = 7200
chikungunya: case targets = 7280, spike labels = 7200
influenza: case targets = 7280, spike labels = 7200
typhoid: case targets = 7280, spike labels = 7200
acute_diarrheal_disease: case targets = 7280, spike labels = 7200

Latest dates in dataset:


,Rows_per_date
date,
2025-12-01,20
2025-12-08,20
2025-12-15,20
2025-12-22,20
2025-12-29,20



Spike label values:
dengue {0.0: 6333, 1.0: 867, nan: 100}
malaria {0.0: 6305, 1.0: 895, nan: 100}
chikungunya {0.0: 6704, 1.0: 496, nan: 100}
influenza {0.0: 6269, 1.0: 931, nan: 100}
typhoid {0.0: 6372, 1.0: 828, nan: 100}
acute_diarrheal_disease {0.0: 6287, 1.0: 913, nan: 100}


In [6]:
# Check whether all expected target columns exist
for disease in diseases:
    expected_cols = [
        f"target_{disease}_cases_next_week",
        f"{disease}_spike_next_week"
    ]

    for col in expected_cols:
        if col not in df.columns:
            print("Missing expected column:", col)

In [7]:

# CELL 4A: Verify spike-label distribution

for disease in diseases:
    col = f"{disease}_spike_next_week"

    print(f"\n{disease}")
    print(df[col].value_counts(dropna=False).sort_index())

    print("Missing labels:", df[col].isna().sum())
    print("Valid labels:", df[col].notna().sum())



dengue
dengue_spike_next_week
0.0    6333
1.0     867
NaN     100
Name: count, dtype: int64
Missing labels: 100
Valid labels: 7200

malaria
malaria_spike_next_week
0.0    6305
1.0     895
NaN     100
Name: count, dtype: int64
Missing labels: 100
Valid labels: 7200

chikungunya
chikungunya_spike_next_week
0.0    6704
1.0     496
NaN     100
Name: count, dtype: int64
Missing labels: 100
Valid labels: 7200

influenza
influenza_spike_next_week
0.0    6269
1.0     931
NaN     100
Name: count, dtype: int64
Missing labels: 100
Valid labels: 7200

typhoid
typhoid_spike_next_week
0.0    6372
1.0     828
NaN     100
Name: count, dtype: int64
Missing labels: 100
Valid labels: 7200

acute_diarrheal_disease
acute_diarrheal_disease_spike_next_week
0.0    6287
1.0     913
NaN     100
Name: count, dtype: int64
Missing labels: 100
Valid labels: 7200


In [8]:
# possible_date_columns = [
#     "date",
#     "week_start_date",
#     "week_start",
#     "report_date",
#     "date_parsed"
# ]

# DATE_COL = next(
#     (col for col in possible_date_columns if col in df.columns),
#     None
# )

# if DATE_COL is None:
#     raise ValueError(
#         "Date column nahi mila. Cell 4 ke column names check karo."
#     )

# df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")

# df = (
#     df.dropna(subset=[DATE_COL])
#       .sort_values(DATE_COL)
#       .reset_index(drop=True)
# )

# print("Date column:", DATE_COL)
# print("First date:", df[DATE_COL].min())
# print("Last date:", df[DATE_COL].max())
# print("Total rows:", len(df))
# print("Unique dates:", df[DATE_COL].nunique())


# CELL 8: Validate and sort dates

DATE_COL = "date"

if DATE_COL not in df.columns:
    raise ValueError("Dataset mein 'date' column nahi mila.")

df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")

# Check invalid dates before removing anything
invalid_dates = df[DATE_COL].isna().sum()
print("Invalid dates:", invalid_dates)

if invalid_dates > 0:
    print("Warning: Invalid-date rows ko investigate karna chahiye.")

df = (
    df.sort_values(DATE_COL)
      .reset_index(drop=True)
)

print("Date column:", DATE_COL)
print("First date:", df[DATE_COL].min())
print("Last date:", df[DATE_COL].max())
print("Total rows:", len(df))
print("Unique dates:", df[DATE_COL].nunique())


Invalid dates: 0
Date column: date
First date: 2019-01-07 00:00:00
Last date: 2025-12-29 00:00:00
Total rows: 7300
Unique dates: 365


In [9]:
diseases = {
    "dengue": {
        "cases_target": "target_dengue_cases_next_week",
        "spike_target": "dengue_spike_next_week"
    },
    "malaria": {
        "cases_target": "target_malaria_cases_next_week",
        "spike_target": "malaria_spike_next_week"
    },
    "chikungunya": {
        "cases_target": "target_chikungunya_cases_next_week",
        "spike_target": "chikungunya_spike_next_week"
    },
    "influenza": {
        "cases_target": "target_influenza_cases_next_week",
        "spike_target": "influenza_spike_next_week"
    },
    "typhoid": {
        "cases_target": "target_typhoid_cases_next_week",
        "spike_target": "typhoid_spike_next_week"
    },
    "acute_diarrheal_disease": {
        "cases_target": "target_acute_diarrheal_disease_cases_next_week",
        "spike_target": "acute_diarrheal_disease_spike_next_week"
    }
}

for disease, targets in diseases.items():
    for task, target in targets.items():
        if target not in df.columns:
            print(f"MISSING: {disease} | {task} | {target}")

print("\nTarget verification completed.")



Target verification completed.


In [10]:
###
missing_targets = [
    f"{disease} | {task} | {target}"
    for disease, targets in diseases.items()
    for task, target in targets.items()
    if target not in df.columns
]

if missing_targets:
    raise ValueError(
        "Required target columns missing:\n"
        + "\n".join(missing_targets)
    )

print("All disease target columns verified successfully!")

All disease target columns verified successfully!


In [11]:

#: Improved calendar features

df["calendar_year"] = df[DATE_COL].dt.year

week_num = df[DATE_COL].dt.isocalendar().week.astype(int)
month_num = df[DATE_COL].dt.month

# Cyclic encoding: December/January and week 52/1
# are close in the seasonal cycle.
df["week_sin"] = np.sin(2 * np.pi * week_num / 52.18)
df["week_cos"] = np.cos(2 * np.pi * week_num / 52.18)

df["month_sin"] = np.sin(2 * np.pi * (month_num - 1) / 12)
df["month_cos"] = np.cos(2 * np.pi * (month_num - 1) / 12)

weather_keywords = [
    "temp", "humidity", "rain", "precip", "wind"
]

weather_cols = [
    col for col in df.columns
    if any(word in col.lower() for word in weather_keywords)
]

location_cols = [
    col for col in df.columns
    if any(word in col.lower() for word in [
        "location", "area", "locality", "city",
        "latitude", "longitude", "district"
    ])
]

calendar_cols = [
    "calendar_year",
    "week_sin", "week_cos",
    "month_sin", "month_cos"
]

print("Weather features:", weather_cols)
print("Location features:", location_cols)
print("Calendar features:", calendar_cols)


Weather features: ['temperature_avg_c', 'humidity_avg_pct', 'rainfall_mm']
Location features: ['city', 'location', 'latitude', 'longitude']
Calendar features: ['calendar_year', 'week_sin', 'week_cos', 'month_sin', 'month_cos']


In [12]:
def get_features_for_disease(disease):
    disease_case_cols = [
        col for col in df.columns
        if col.lower().startswith(disease + "_")
        and "case" in col.lower()
        and not any(word in col.lower() for word in [
            "target", "spike", "hospital", "death"
        ])
    ]

    candidates = (
        disease_case_cols
        + weather_cols
        + location_cols
        + calendar_cols
    )

    features = list(dict.fromkeys(candidates))

    # Strictly exclude future targets and spike labels
    features = [
        col for col in features
        if col in df.columns
        and col != DATE_COL
        and not col.lower().startswith("target_")
        and "next_week" not in col.lower()
        and "spike" not in col.lower()
    ]

    if not disease_case_cols:
        raise ValueError(
            f"{disease} ke case features nahi mile. "
            "Cell 4 mein column names check karo."
        )

    return features


for disease in diseases:
    print(f"\n{disease.upper()}")
    print(get_features_for_disease(disease))


DENGUE
['dengue_cases', 'dengue_cases_lag_1w', 'dengue_cases_lag_2w', 'dengue_cases_lag_4w', 'dengue_cases_rolling_mean_4w', 'temperature_avg_c', 'humidity_avg_pct', 'rainfall_mm', 'city', 'location', 'latitude', 'longitude', 'calendar_year', 'week_sin', 'week_cos', 'month_sin', 'month_cos']

MALARIA
['malaria_cases', 'malaria_cases_lag_1w', 'malaria_cases_lag_2w', 'malaria_cases_lag_4w', 'malaria_cases_rolling_mean_4w', 'temperature_avg_c', 'humidity_avg_pct', 'rainfall_mm', 'city', 'location', 'latitude', 'longitude', 'calendar_year', 'week_sin', 'week_cos', 'month_sin', 'month_cos']

CHIKUNGUNYA
['chikungunya_cases', 'chikungunya_cases_lag_1w', 'chikungunya_cases_lag_2w', 'chikungunya_cases_lag_4w', 'chikungunya_cases_rolling_mean_4w', 'temperature_avg_c', 'humidity_avg_pct', 'rainfall_mm', 'city', 'location', 'latitude', 'longitude', 'calendar_year', 'week_sin', 'week_cos', 'month_sin', 'month_cos']

INFLUENZA
['influenza_cases', 'influenza_cases_lag_1w', 'influenza_cases_lag_2w',

In [13]:

# DATA LEAKAGE & DATA QUALITY AUDIT

print("=" * 65)
print("DATA LEAKAGE & DATA QUALITY AUDIT")
print("=" * 65)

# 1. Basic data integrity
print("\n1. DATA INTEGRITY")
print("Dataset shape:", df.shape)
print("Duplicate full rows:", df.duplicated().sum())
print("Missing dates:", df[DATE_COL].isna().sum())

if "location" in df.columns:
    duplicate_keys = df.duplicated(
        subset=["location", DATE_COL], keep=False
    )
    print("Duplicate location-date rows:", int(duplicate_keys.sum()))

# Sort each location's history chronologically
audit_df = df.sort_values(
    ["location", DATE_COL] if "location" in df.columns
    else [DATE_COL]
).copy()

# 2. Verify that model features exclude future targets
print("\n2. FEATURE LEAKAGE CHECK")

leak_keywords = ["target_", "next_week", "spike"]
feature_leaks = []

for disease in diseases:
    features = get_features_for_disease(disease)

    suspicious = [
        col for col in features
        if any(word in col.lower() for word in leak_keywords)
    ]

    print(f"{disease}: {len(features)} features")
    if suspicious:
        feature_leaks.extend(
            [(disease, col) for col in suspicious]
        )
        print("  REVIEW REQUIRED:", suspicious)
    else:
        print("  PASS: no obvious target/spike columns in features")

if feature_leaks:
    print("\nWARNING: Review these suspicious features:", feature_leaks)
else:
    print("\nNo obvious future-target columns found in model features.")

# 3. Check lag features against each location's previous rows
print("\n3. LAG FEATURE ALIGNMENT")

group_cols = ["location"] if "location" in audit_df.columns else []

for disease in diseases:
    for lag in [1, 2, 4]:
        source_col = f"{disease}_cases"
        lag_col = f"{disease}_cases_lag_{lag}w"

        if source_col not in audit_df or lag_col not in audit_df:
            continue

        if group_cols:
            expected = audit_df.groupby(
                group_cols, sort=False
            )[source_col].shift(lag)
        else:
            expected = audit_df[source_col].shift(lag)

        actual = pd.to_numeric(audit_df[lag_col], errors="coerce")
        expected = pd.to_numeric(expected, errors="coerce")

        valid = actual.notna() & expected.notna()
        mismatches = int(
            (~np.isclose(
                actual[valid],
                expected[valid],
                rtol=1e-5,
                atol=1e-5
            )).sum()
        )

        print(
            f"{lag_col}: compared={int(valid.sum())}, "
            f"mismatches={mismatches}"
        )

# 4. Check the meaning of rolling 4-week averages
print("\n4. ROLLING FEATURE CHECK")

for disease in diseases:
    source_col = f"{disease}_cases"
    roll_col = f"{disease}_cases_rolling_mean_4w"

    if source_col not in audit_df or roll_col not in audit_df:
        continue

    if group_cols:
        grouped = audit_df.groupby(group_cols, sort=False)[source_col]
        expected_inclusive = grouped.transform(
            lambda s: s.rolling(4, min_periods=4).mean()
        )
        expected_past_only = grouped.transform(
            lambda s: s.shift(1).rolling(4, min_periods=4).mean()
        )
    else:
        expected_inclusive = audit_df[source_col].rolling(
            4, min_periods=4
        ).mean()
        expected_past_only = audit_df[source_col].shift(
            1
        ).rolling(4, min_periods=4).mean()

    actual = pd.to_numeric(audit_df[roll_col], errors="coerce")

    def count_mismatches(a, b):
        valid = actual.notna() & b.notna()
        if valid.sum() == 0:
            return None
        return int(
            (~np.isclose(
                actual[valid], b[valid],
                rtol=1e-5, atol=1e-5
            )).sum()
        )

    print(
        f"{roll_col}: "
        f"current-inclusive mismatches="
        f"{count_mismatches(actual, expected_inclusive)}, "
        f"past-only mismatches="
        f"{count_mismatches(actual, expected_past_only)}"
    )

# 5. Spike label distribution and missing values
print("\n5. SPIKE LABEL DISTRIBUTION")

for disease, targets in diseases.items():
    col = targets["spike_target"]
    labels = pd.to_numeric(df[col], errors="coerce")
    valid_labels = labels[labels.isin([0, 1])]

    print(f"\n{disease}:")
    print("  Missing/invalid labels:", int(labels.isna().sum() +
                                              (~labels.isin([0, 1]) & labels.notna()).sum()))
    print("  Valid labels:", len(valid_labels))
    print("  Spike prevalence:", round(float(valid_labels.mean()), 4)
          if len(valid_labels) else np.nan)
    print("  Label counts:", valid_labels.value_counts().to_dict())



# 6. Verify next-week target alignment

print("\n6. NEXT-WEEK TARGET ALIGNMENT")

target_audit_failed = False

target_audit_df = df.sort_values(
    ["location", DATE_COL]
    if "location" in df.columns
    else [DATE_COL]
).copy()


# Check weekly date gaps
if "location" in target_audit_df.columns:
    week_gaps = (
        target_audit_df.groupby("location")[DATE_COL]
        .diff()
        .dropna()
    )
else:
    week_gaps = target_audit_df[DATE_COL].diff().dropna()

bad_gaps = week_gaps[
    week_gaps != pd.Timedelta(days=7)
]

print("Non-weekly date gaps:", len(bad_gaps))

if len(bad_gaps) > 0:
    print("WARNING: Some weekly date gaps are not 7 days.")
    target_audit_failed = True


for disease, targets in diseases.items():
    source_col = f"{disease}_cases"
    target_col = targets["cases_target"]

    if source_col not in target_audit_df.columns:
        print(f"{disease}: source column missing")
        target_audit_failed = True
        continue

    expected = (
        target_audit_df.groupby("location")[source_col].shift(-1)
        if "location" in target_audit_df.columns
        else target_audit_df[source_col].shift(-1)
    )

    actual = pd.to_numeric(
        target_audit_df[target_col],
        errors="coerce"
    )

    expected = pd.to_numeric(expected, errors="coerce")
    valid = expected.notna() & actual.notna()

    missing_pattern_mismatches = int(
        (expected.isna() != actual.isna()).sum()
    )

    print(
        f"{disease}: missing-pattern mismatches="
        f"{missing_pattern_mismatches}"
    )

    if missing_pattern_mismatches > 0:
        target_audit_failed = True

    mismatch_count = int(
        (
            ~np.isclose(
                expected.loc[valid],
                actual.loc[valid],
                rtol=1e-5,
                atol=1e-5
            )
        ).sum()
    )

    print(
        f"{disease}: "
        f"compared={int(valid.sum())}, "
        f"mismatches={mismatch_count}"
    )

    if mismatch_count > 0:
        target_audit_failed = True

        

if target_audit_failed:
    raise ValueError(
        "Target alignment failed. Training se pehle "
        "target generation aur date order check karo."
    )

print("Target alignment audit passed.")
 

print("\nAUDIT COMPLETE.")
print("Review every mismatch and warning before trusting model results.")


DATA LEAKAGE & DATA QUALITY AUDIT

1. DATA INTEGRITY
Dataset shape: (7300, 84)
Duplicate full rows: 0
Missing dates: 0
Duplicate location-date rows: 0

2. FEATURE LEAKAGE CHECK
dengue: 17 features
  PASS: no obvious target/spike columns in features
malaria: 17 features
  PASS: no obvious target/spike columns in features
chikungunya: 17 features
  PASS: no obvious target/spike columns in features
influenza: 17 features
  PASS: no obvious target/spike columns in features
typhoid: 17 features
  PASS: no obvious target/spike columns in features
acute_diarrheal_disease: 17 features
  PASS: no obvious target/spike columns in features

No obvious future-target columns found in model features.

3. LAG FEATURE ALIGNMENT
dengue_cases_lag_1w: compared=7280, mismatches=0
dengue_cases_lag_2w: compared=7260, mismatches=0
dengue_cases_lag_4w: compared=7220, mismatches=0
malaria_cases_lag_1w: compared=7280, mismatches=0
malaria_cases_lag_2w: compared=7260, mismatches=0
malaria_cases_lag_4w: compared=7

In [14]:
unique_dates = np.sort(df[DATE_COL].dropna().unique())

if len(unique_dates) < 10:
    raise ValueError("Time-based split ke liye kaafi dates nahi hain.")

cutoff_index = int(len(unique_dates) * 0.80)
cutoff_index = min(
    max(cutoff_index, 1),
    len(unique_dates) - 1
)

TEST_START_DATE = pd.Timestamp(unique_dates[cutoff_index])

print("Training dates before:", TEST_START_DATE)
print("Testing dates from:", TEST_START_DATE)
print("Total unique dates:", len(unique_dates))

Training dates before: 2024-08-12 00:00:00
Testing dates from: 2024-08-12 00:00:00
Total unique dates: 365


In [15]:

def make_pipeline(X_train, task):
    # 1. Validate task
    if task not in ("regression", "classification"):
        raise ValueError(
            "task must be 'regression' or 'classification'"
        )

    if X_train.empty or X_train.shape[1] == 0:
        raise ValueError("X_train is empty or has no feature columns.")

    # 2. Identify categorical and numeric features
    categorical_cols = X_train.select_dtypes(
        include=["object", "category", "bool", "string"]
    ).columns.tolist()

    numeric_cols = X_train.select_dtypes(
        include=["number"]
    ).columns.tolist()

    # Unexpected dtypes should not silently enter the model
    unsupported_cols = [
        col for col in X_train.columns
        if col not in categorical_cols and col not in numeric_cols
    ]

    if unsupported_cols:
        raise ValueError(
            f"Unsupported feature columns: {unsupported_cols}. "
            "Remove them or convert them appropriately first."
        )

    transformers = []

    # 3. Numeric preprocessing
    if numeric_cols:
        numeric_pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="median"))
        ])

        transformers.append(
            ("numeric", numeric_pipeline, numeric_cols)
        )

    # 4. Categorical preprocessing
    if categorical_cols:
        categorical_pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ])

        transformers.append(
            ("categorical", categorical_pipeline, categorical_cols)
        )

    preprocess = ColumnTransformer(
        transformers=transformers,
        remainder="drop"
    )

    # 5. Select model for the task
    if task == "regression":
        estimator = RandomForestRegressor(
            n_estimators=200,
            min_samples_leaf=2,
            random_state=42,
            n_jobs=-1
        )
    else:
        estimator = RandomForestClassifier(
            n_estimators=250,
            min_samples_leaf=2,
            class_weight="balanced_subsample",
            random_state=42,
            n_jobs=-1
        )

    # 6. Return complete pipeline
    return Pipeline([
        ("preprocess", preprocess),
        ("model", estimator)
    ])


print("Disease prediction pipeline ready.")


Disease prediction pipeline ready.


In [16]:

# CELL: Tune regression model using chronological validation

from sklearn.metrics import mean_absolute_error


# Candidate Random Forest configurations
param_grid = [
    {"n_estimators": 200, "max_depth": 10, "min_samples_leaf": 2},
    {"n_estimators": 300, "max_depth": 15, "min_samples_leaf": 2},
    {"n_estimators": 300, "max_depth": None, "min_samples_leaf": 3},
    {"n_estimators": 400, "max_depth": 15, "min_samples_leaf": 4},
]

# Use only dates before the final test period
pretest_dates = np.sort(
    df.loc[
        df[DATE_COL] < TEST_START_DATE, DATE_COL
    ].dropna().unique()
)

if len(pretest_dates) < 10:
    raise ValueError(
        "Not enough pre-test dates for chronological validation."
    )

# Use the earlier 80% of pre-test dates for fitting,
# and the later portion for validation.
val_index = int(len(pretest_dates) * 0.80)
val_index = min(max(val_index, 1), len(pretest_dates) - 1)

val_start = pd.Timestamp(pretest_dates[val_index])

best_params = {}
tuning_results = []

for disease, targets in diseases.items():

    features = get_features_for_disease(disease)
    target_col = targets["cases_target"]

    # Convert target to numeric; keep missing targets as missing
    y = pd.to_numeric(df[target_col], errors="coerce")

    # Fit data: before validation period, with a 7-day gap
    fit_mask = (
        (df[DATE_COL] < val_start - pd.Timedelta(days=7))
        & y.notna()
    )

    # Validation data: before the final test period,
    # with a 7-day boundary gap to avoid target overlap
    val_mask = (
        (df[DATE_COL] >= val_start)
        & (
            df[DATE_COL]
            < TEST_START_DATE - pd.Timedelta(days=7)
        )
        & y.notna()
    )

    X_fit = df.loc[fit_mask, features]
    y_fit = y.loc[fit_mask]

    X_val = df.loc[val_mask, features]
    y_val = y.loc[val_mask]

    if len(X_fit) == 0 or len(X_val) == 0:
        raise ValueError(
            f"{disease}: empty train/validation split. "
            "Check TEST_START_DATE and validation boundaries."
        )

    if y_fit.nunique() < 2:
        raise ValueError(
            f"{disease}: insufficient target variation in fit data."
        )

    best_mae = float("inf")
    chosen_params = None

    for params in param_grid:

        model = make_pipeline(X_fit, "regression")

        model.set_params(
            model__n_estimators=params["n_estimators"],
            model__max_depth=params["max_depth"],
            model__min_samples_leaf=params["min_samples_leaf"]
        )

        model.fit(X_fit, y_fit)

        # Case counts cannot be negative
        pred = np.maximum(model.predict(X_val), 0)

        score = mean_absolute_error(y_val, pred)

        tuning_results.append({
            "disease": disease,
            **params,
            "validation_MAE": score,
            "fit_rows": len(X_fit),
            "validation_rows": len(X_val)
        })

        if score < best_mae:
            best_mae = score
            chosen_params = params.copy()

    best_params[disease] = chosen_params

    print(
        f"{disease}: validation MAE = {best_mae:.3f}; "
        f"best parameters = {chosen_params}"
    )

tuning_df = pd.DataFrame(tuning_results)

display(
    tuning_df.sort_values(
        ["disease", "validation_MAE"]
    ).round(3)
)

print("\nBest parameters for each disease:")
print(best_params)


dengue: validation MAE = 3.261; best parameters = {'n_estimators': 200, 'max_depth': 10, 'min_samples_leaf': 2}
malaria: validation MAE = 2.646; best parameters = {'n_estimators': 300, 'max_depth': None, 'min_samples_leaf': 3}
chikungunya: validation MAE = 1.760; best parameters = {'n_estimators': 200, 'max_depth': 10, 'min_samples_leaf': 2}
influenza: validation MAE = 2.878; best parameters = {'n_estimators': 200, 'max_depth': 10, 'min_samples_leaf': 2}
typhoid: validation MAE = 2.216; best parameters = {'n_estimators': 300, 'max_depth': 15, 'min_samples_leaf': 2}
acute_diarrheal_disease: validation MAE = 2.701; best parameters = {'n_estimators': 200, 'max_depth': 10, 'min_samples_leaf': 2}


,disease,n_estimators,max_depth,min_samples_leaf,validation_MAE,fit_rows,validation_rows
20,acute_diarrheal_disease,200,10.0,2,2.701,4640,1160
21,acute_diarrheal_disease,300,15.0,2,2.725,4640,1160
22,acute_diarrheal_disease,300,NaN,3,2.726,4640,1160
23,acute_diarrheal_disease,400,15.0,4,2.727,4640,1160
8,chikungunya,200,10.0,2,1.760,4640,1160
10,chikungunya,300,NaN,3,1.762,4640,1160
9,chikungunya,300,15.0,2,1.763,4640,1160
11,chikungunya,400,15.0,4,1.764,4640,1160
0,dengue,200,10.0,2,3.261,4640,1160
3,dengue,400,15.0,4,3.281,4640,1160



Best parameters for each disease:
{'dengue': {'n_estimators': 200, 'max_depth': 10, 'min_samples_leaf': 2}, 'malaria': {'n_estimators': 300, 'max_depth': None, 'min_samples_leaf': 3}, 'chikungunya': {'n_estimators': 200, 'max_depth': 10, 'min_samples_leaf': 2}, 'influenza': {'n_estimators': 200, 'max_depth': 10, 'min_samples_leaf': 2}, 'typhoid': {'n_estimators': 300, 'max_depth': 15, 'min_samples_leaf': 2}, 'acute_diarrheal_disease': {'n_estimators': 200, 'max_depth': 10, 'min_samples_leaf': 2}}


In [17]:

OUTPUT_DIR = Path("trained_models")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

all_models = {}
metrics_rows = []
feature_metadata = {}
prediction_rows = []
spike_thresholds = {}

for disease, targets in diseases.items():

    print(f"\n{'=' * 55}")
    print("Training:", disease)
    print("=" * 55)

    features = get_features_for_disease(disease)
    feature_metadata[disease] = features

   
    # A. REGRESSION: Predict next week's cases
  
    case_target = targets["cases_target"]
    y_cases = pd.to_numeric(df[case_target], errors="coerce")

    train_mask = (
    (df[DATE_COL] < TEST_START_DATE - pd.Timedelta(days=7))
    & y_cases.notna()
    )

    test_mask = (
        (df[DATE_COL] >= TEST_START_DATE)
        & y_cases.notna()
    )

    X_train = df.loc[train_mask, features]
    y_train = y_cases.loc[train_mask]

    X_test = df.loc[test_mask, features]
    y_test = y_cases.loc[test_mask]

    if X_train.empty or X_test.empty:
        raise ValueError(
            f"{disease}: regression train/test data is empty."
        )

    if y_train.empty or y_test.empty:
        raise ValueError(
            f"{disease}: regression target data is empty."
        )
        
    reg_model = make_pipeline(X_train, "regression")
    
    chosen = best_params[disease]
    
    reg_model.set_params(
        model__n_estimators=chosen["n_estimators"],
        model__max_depth=chosen["max_depth"],
        model__min_samples_leaf=chosen["min_samples_leaf"]
    )
    
    reg_model.fit(X_train, y_train)




    case_predictions = np.maximum(
        reg_model.predict(X_test), 0
    )

    mae = mean_absolute_error(y_test, case_predictions)

    
    # Additional regression evaluation
    rmse = np.sqrt(
        mean_squared_error(y_test, case_predictions)
    )
    
        
    r2 = r2_score(y_test, case_predictions)
    
    
    print("MAE:", round(mae, 3))
    print("RMSE:", round(rmse, 3))
    print("R2:", round(r2, 3))


    # Baseline: next week's cases = current week's cases
    current_case_col = f"{disease}_cases"
    
    if current_case_col in df.columns:
        baseline = pd.to_numeric(
            df.loc[test_mask, current_case_col],
            errors="coerce"
        )
    
        baseline_valid = baseline.notna()
    
        baseline_mae = (
            mean_absolute_error(
                y_test.loc[baseline_valid],
                baseline.loc[baseline_valid]
            )
            if baseline_valid.any()
            else np.nan
        )
    else:
        baseline_mae = np.nan

    # Calculate improvement only after baseline MAE is available
    baseline_improvement = (
        ((baseline_mae - mae) / baseline_mae) * 100
        if pd.notna(baseline_mae) and baseline_mae > 0
        else np.nan
    )
    
    print(
        "Baseline improvement:",
        f"{baseline_improvement:.2f}%"
        if pd.notna(baseline_improvement)
        else "N/A"
    )
    
    
    
    print("Cases MAE:", round(mae, 3))
    print(
        "Baseline MAE:",
        round(baseline_mae, 3)
        if pd.notna(baseline_mae)
        else "N/A"
    )

    reg_path = OUTPUT_DIR / f"{disease}_regressor.pkl"
    joblib.dump(reg_model, reg_path)

    all_models[f"{disease}_regressor"] = reg_model

    
    metrics_rows.append({
        "disease": disease,
        "task": "cases_regression",
        "train_rows": len(X_train),
        "test_rows": len(X_test),
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "baseline_MAE": baseline_mae,
        "baseline_improvement_pct": baseline_improvement,
        "precision": np.nan,
        "recall": np.nan,
        "F1": np.nan,
        "average_precision": np.nan
    })




    print("Cases MAE:", round(mae, 3))
    print("Baseline MAE:", round(baseline_mae, 3))
    print("Saved:", reg_path)

    
    for idx, actual, predicted in zip(
        X_test.index, y_test, case_predictions
    ):
        prediction_rows.append({
            "date": df.loc[idx, DATE_COL],
            "forecast_for_date": (
                pd.Timestamp(df.loc[idx, DATE_COL])
                + pd.Timedelta(days=7)
            ),
            "location": (
                df.loc[idx, "location"]
                if "location" in df.columns
                else "unknown"
            ),
            "disease": disease,
            "task": "cases",
            "actual": actual,
            "predicted": predicted,
            "spike_probability": np.nan
        })




    # B. CLASSIFICATION: Predict spike risk

    spike_target = targets["spike_target"]
    y_spike = pd.to_numeric(
        df[spike_target], errors="coerce"
    )

    train_spike_mask = (
    (df[DATE_COL] < TEST_START_DATE - pd.Timedelta(days=7))
    & y_spike.isin([0, 1])
    )
    
    test_spike_mask = (
        (df[DATE_COL] >= TEST_START_DATE)
        & y_spike.isin([0, 1])
    )

    X_train_s = df.loc[train_spike_mask, features]
    y_train_s = y_spike.loc[train_spike_mask].astype(int)

    X_test_s = df.loc[test_spike_mask, features]
    y_test_s = y_spike.loc[test_spike_mask].astype(int)

    if X_train_s.empty or X_test_s.empty:
        raise ValueError(
            f"{disease}: classification train/test data is empty."
        )

    if y_train_s.nunique() < 2:
        raise ValueError(
            f"{disease}: training data must contain both "
            "0 (no spike) and 1 (spike)."
        )
    
  
    # B1. Chronological train-validation split

    train_dates = np.sort(
        df.loc[train_spike_mask, DATE_COL].unique()
    )

    if len(train_dates) < 3:
        raise ValueError(
            f"{disease}: validation split ke liye "
            "sufficient dates nahi hain."
        )

    val_start_index = int(len(train_dates) * 0.80)
    val_start_index = min(
        max(val_start_index, 1),
        len(train_dates) - 1
    )

    val_start_date = pd.Timestamp(
        train_dates[val_start_index]
    )

    val_train_mask = (
        train_spike_mask
        & (df[DATE_COL] < val_start_date - pd.Timedelta(days=7))
    )

    val_mask = (
        (df[DATE_COL] >= val_start_date)
        & (df[DATE_COL] < TEST_START_DATE - pd.Timedelta(days=7))
        & y_spike.isin([0, 1])
    )

    X_fit_val = df.loc[val_train_mask, features]
    y_fit_val = y_spike.loc[val_train_mask].astype(int)

    X_val = df.loc[val_mask, features]
    y_val = y_spike.loc[val_mask].astype(int)

    if (
        y_fit_val.nunique() < 2
        or y_val.nunique() < 2
    ):
        raise ValueError(
            f"{disease}: fit ya validation data mein "
            "0 aur 1 dono labels hone chahiye."
        )


    # B2. Choose threshold using validation data

    validation_model = make_pipeline(
        X_fit_val, "classification"
    )
    validation_model.fit(X_fit_val, y_fit_val)

    val_probabilities = validation_model.predict_proba(X_val)
    val_classes = validation_model.named_steps["model"].classes_
    val_spike_index = list(val_classes).index(1)

    val_spike_probabilities = val_probabilities[
        :, val_spike_index
    ]

    best_threshold = 0.50
    best_f2 = -1.0

    for threshold in np.arange(0.05, 0.81, 0.01):

        val_predictions = (
            val_spike_probabilities >= threshold
        ).astype(int)

        val_precision = precision_score(
            y_val, val_predictions, zero_division=0
        )

        val_recall = recall_score(
            y_val, val_predictions, zero_division=0
        )

        denominator = 4 * val_precision + val_recall

        f2 = (
            5 * val_precision * val_recall / denominator
            if denominator > 0
            else 0.0
        )

        if f2 > best_f2:
            best_f2 = f2
            best_threshold = float(threshold)

    spike_thresholds[disease] = best_threshold


    # B3. Train final classifier on pre-test data

    clf_model = make_pipeline(
        X_train_s, "classification"
    )
    clf_model.fit(X_train_s, y_train_s)

    # Evaluate only on held-out test data
    probabilities = clf_model.predict_proba(X_test_s)

    class_labels = clf_model.named_steps["model"].classes_
    spike_index = list(class_labels).index(1)

    spike_probabilities = probabilities[:, spike_index]

    spike_predictions = (
        spike_probabilities >= best_threshold
    ).astype(int)

    print(
        "Selected spike threshold:",
        round(best_threshold, 2)
    )
    print("Validation F2:", round(best_f2, 3))


    # B4. Test metrics

    precision = precision_score(
        y_test_s, spike_predictions, zero_division=0
    )

    recall = recall_score(
        y_test_s, spike_predictions, zero_division=0
    )

    f1 = f1_score(
        y_test_s, spike_predictions, zero_division=0
    )

    if y_test_s.nunique() == 2:
        avg_precision = average_precision_score(
            y_test_s, spike_probabilities
        )
    else:
        avg_precision = np.nan

    # B4 EXTRA: Confusion matrix and detailed test report


    cm = confusion_matrix(
        y_test_s,
        spike_predictions,
        labels=[0, 1]
    )

    tn, fp, fn, tp = cm.ravel()

    test_prevalence = float(y_test_s.mean())

    print("\n--- Detailed test evaluation ---")
    print("Test spike prevalence:", round(test_prevalence, 3))
    print("True negatives:", tn)
    print("False positives (false alerts):", fp)
    print("False negatives (missed spikes):", fn)
    print("True positives:", tp)

    print("\nClassification report:")
    print(classification_report(
        y_test_s,
        spike_predictions,
        labels=[0, 1],
        target_names=["No spike", "Spike"],
        zero_division=0
    ))



    clf_path = OUTPUT_DIR / f"{disease}_spike_classifier.pkl"
    joblib.dump(clf_model, clf_path)

    all_models[f"{disease}_spike_classifier"] = clf_model

    
    metrics_rows.append({
        "disease": disease,
        "task": "spike_classification",
        "train_rows": len(X_train_s),
        "test_rows": len(X_test_s),
        "MAE": np.nan,
        "RMSE": np.nan,
        "R2": np.nan,
        "baseline_MAE": np.nan,
        "baseline_improvement_pct": np.nan,
        "precision": precision,
        "recall": recall,
        "F1": f1,
        "average_precision": avg_precision,
        "threshold": best_threshold,
        "validation_F2": best_f2,
        "true_negatives": int(tn),
        "false_positives": int(fp),
        "false_negatives": int(fn),
        "true_positives": int(tp),
        "test_prevalence": test_prevalence
    })



    print("Spike precision:", round(precision, 3))
    print("Spike recall:", round(recall, 3))
    print("Spike F1:", round(f1, 3))
    print("Average precision:", round(avg_precision, 3))
    print("Saved:", clf_path)

    for idx, actual, predicted, probability in zip(
        X_test_s.index,
        y_test_s,
        spike_predictions,
        spike_probabilities
    ):
        prediction_rows.append({
            
            "date": df.loc[idx, DATE_COL],
            "forecast_for_date": (
                pd.Timestamp(df.loc[idx, DATE_COL])
                + pd.Timedelta(days=7)
            ),

            "location": (
                df.loc[idx, "location"]
                if "location" in df.columns
                else "unknown"
            ),
            "disease": disease,
            "task": "spike",
            "actual": actual,
            "predicted": predicted,
            "spike_probability": probability
        })

print("\nAll 12 models trained and saved!")


# Save combined evaluation results
metrics_df = pd.DataFrame(metrics_rows)
metrics_df.to_csv(
    OUTPUT_DIR / "model_evaluation_metrics.csv",
    index=False
)

predictions_df = pd.DataFrame(prediction_rows)
predictions_df.to_csv(
    OUTPUT_DIR / "test_predictions.csv",
    index=False
)

# Save metadata for demo integration
metadata = {
    "project": "Pune Multi-Disease Prediction & Outbreak Alert System",
    "dataset_type": "synthetic",
    "date_column": DATE_COL,
    "test_start_date": str(pd.Timestamp(TEST_START_DATE).date()),
    "features": feature_metadata,
    "spike_thresholds": spike_thresholds,
    "regression_parameters": best_params
}

with open(
    OUTPUT_DIR / "model_metadata.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(metadata, f, indent=4)

print("\nAll models, metrics, predictions and metadata saved!")
print("Output folder:", OUTPUT_DIR.resolve())



Training: dengue
MAE: 3.154
RMSE: 4.165
R2: 0.672
Baseline improvement: 24.20%
Cases MAE: 3.154
Baseline MAE: 4.16
Cases MAE: 3.154
Baseline MAE: 4.16
Saved: trained_models\dengue_regressor.pkl
Selected spike threshold: 0.21
Validation F2: 0.532

--- Detailed test evaluation ---
Test spike prevalence: 0.098
True negatives: 1080
False positives (false alerts): 219
False negatives (missed spikes): 58
True positives: 83

Classification report:
              precision    recall  f1-score   support

    No spike       0.95      0.83      0.89      1299
       Spike       0.27      0.59      0.37       141

    accuracy                           0.81      1440
   macro avg       0.61      0.71      0.63      1440
weighted avg       0.88      0.81      0.84      1440

Spike precision: 0.275
Spike recall: 0.589
Spike F1: 0.375
Average precision: 0.329
Saved: trained_models\dengue_spike_classifier.pkl

Training: malaria
MAE: 2.793
RMSE: 3.639
R2: 0.544
Baseline improvement: 25.18%
Cases MAE: 2

In [18]:
metrics_df = pd.DataFrame(metrics_rows)

display(
    metrics_df.round(3)
    .sort_values(["task", "disease"])
    .reset_index(drop=True)
)

metrics_df.to_csv(
    OUTPUT_DIR / "model_metrics.csv",
    index=False
)

print("Metrics saved.")

,disease,task,train_rows,test_rows,MAE,RMSE,R2,baseline_MAE,baseline_improvement_pct,precision,recall,F1,average_precision,threshold,validation_F2,true_negatives,false_positives,false_negatives,true_positives,test_prevalence
0,acute_diarrheal_disease,cases_regression,5820,1440,2.656,3.519,0.512,3.620,26.644,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,chikungunya,cases_regression,5820,1440,1.783,2.334,0.479,2.413,26.112,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,dengue,cases_regression,5820,1440,3.154,4.165,0.672,4.160,24.196,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,influenza,cases_regression,5820,1440,2.695,3.494,0.479,3.659,26.348,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,malaria,cases_regression,5820,1440,2.793,3.639,0.544,3.733,25.176,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,typhoid,cases_regression,5820,1440,2.135,2.744,0.459,2.891,26.136,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,acute_diarrheal_disease,spike_classification,5740,1440,NaN,NaN,NaN,NaN,NaN,0.233,0.655,0.344,0.341,0.18,0.516,891.0,375.0,60.0,114.0,0.121
7,chikungunya,spike_classification,5740,1440,NaN,NaN,NaN,NaN,NaN,0.148,0.585,0.236,0.205,0.14,0.485,1081.0,277.0,34.0,48.0,0.057
8,dengue,spike_classification,5740,1440,NaN,NaN,NaN,NaN,NaN,0.275,0.589,0.375,0.329,0.21,0.532,1080.0,219.0,58.0,83.0,0.098
9,influenza,spike_classification,5740,1440,NaN,NaN,NaN,NaN,NaN,0.219,0.865,0.349,0.316,0.14,0.541,622.0,618.0,27.0,173.0,0.139


Metrics saved.


In [19]:
predictions_df = pd.DataFrame(prediction_rows)

predictions_df.to_csv(
    OUTPUT_DIR / "test_predictions.csv",
    index=False
)

display(predictions_df.head(20))

print("Test predictions saved.")

,date,forecast_for_date,location,disease,task,actual,predicted,spike_probability
0,2024-08-12,2024-08-19,Sinhagad_Road,dengue,cases,30.0,17.391511,NaN
1,2024-08-12,2024-08-19,Lohegaon,dengue,cases,33.0,31.484522,NaN
2,2024-08-12,2024-08-19,Kondhwa,dengue,cases,20.0,19.131360,NaN
3,2024-08-12,2024-08-19,Yerawada,dengue,cases,23.0,21.687903,NaN
4,2024-08-12,2024-08-19,Baner,dengue,cases,21.0,21.155255,NaN
5,2024-08-12,2024-08-19,Hadapsar,dengue,cases,19.0,22.965010,NaN
6,2024-08-12,2024-08-19,Chinchwad,dengue,cases,14.0,19.273754,NaN
7,2024-08-12,2024-08-19,Kothrud,dengue,cases,23.0,18.134333,NaN
8,2024-08-12,2024-08-19,Pimpri,dengue,cases,31.0,25.380684,NaN
9,2024-08-12,2024-08-19,Dhankawadi,dengue,cases,23.0,28.527897,NaN


Test predictions saved.


In [20]:
spike_results = predictions_df[
    predictions_df["task"] == "spike"
].copy()

display(
    spike_results[
        [
            "date",
            "location",
            "disease",
            "actual",
            "predicted",
            "spike_probability"
        ]
    ].head(25)
)

,date,location,disease,actual,predicted,spike_probability
1440,2024-08-12,Sinhagad_Road,dengue,1.0,0.0,0.088291
1441,2024-08-12,Lohegaon,dengue,0.0,0.0,0.029189
1442,2024-08-12,Kondhwa,dengue,0.0,0.0,0.025984
1443,2024-08-12,Yerawada,dengue,0.0,0.0,0.078561
1444,2024-08-12,Baner,dengue,0.0,0.0,0.130936
1445,2024-08-12,Hadapsar,dengue,0.0,0.0,0.060392
1446,2024-08-12,Chinchwad,dengue,0.0,0.0,0.125551
1447,2024-08-12,Kothrud,dengue,0.0,0.0,0.021328
1448,2024-08-12,Pimpri,dengue,0.0,0.0,0.071615
1449,2024-08-12,Dhankawadi,dengue,0.0,0.0,0.038975


In [21]:
metadata = {
    "date_column": DATE_COL,
    "test_start_date": str(TEST_START_DATE.date()),
    "diseases": diseases,
    "features": feature_metadata,
    "spike_thresholds": spike_thresholds
}

with open(OUTPUT_DIR / "model_metadata.json", "w") as file:
    json.dump(metadata, file, indent=4)

print("Metadata saved.")
print("Total models:", len(all_models))
print("Saved folder:", OUTPUT_DIR.resolve())

Metadata saved.
Total models: 12
Saved folder: C:\Users\YASH\Prime Classes\health2\trained_models


In [22]:
disease = "dengue"
features = feature_metadata[disease]

reg_model = joblib.load(
    OUTPUT_DIR / f"{disease}_regressor.pkl"
)

spike_model = joblib.load(
    OUTPUT_DIR / f"{disease}_spike_classifier.pkl"
)

example_mask = (
    (df[DATE_COL] >= TEST_START_DATE)
    & pd.to_numeric(
        df[diseases[disease]["cases_target"]],
        errors="coerce"
    ).notna()
)

example_row = df.loc[example_mask, features].head(1)

if example_row.empty:
    raise ValueError("Example prediction row nahi mila.")

predicted_cases = max(
    0.0,
    float(reg_model.predict(example_row)[0])
)

probabilities = spike_model.predict_proba(example_row)[0]
classes = spike_model.named_steps["model"].classes_
spike_index = list(classes).index(1)

spike_probability = float(probabilities[spike_index])

print("Location:", df.loc[example_row.index[0], "location"])
print("Disease:", disease)
print("Predicted next-week cases:", round(predicted_cases, 2))
print("Estimated spike probability:", round(spike_probability * 100, 2), "%")
print(
    "Spike alert:",
    "YES" if spike_probability >= spike_thresholds.get(disease, 0.50) else "NO"
    
)

Location: Sinhagad_Road
Disease: dengue
Predicted next-week cases: 17.39
Estimated spike probability: 8.83 %
Spike alert: NO


In [23]:
for file in sorted(OUTPUT_DIR.iterdir()):
    print(file.name)

acute_diarrheal_disease_regressor.pkl
acute_diarrheal_disease_spike_classifier.pkl
chikungunya_regressor.pkl
chikungunya_spike_classifier.pkl
dengue_regressor.pkl
dengue_spike_classifier.pkl
influenza_regressor.pkl
influenza_spike_classifier.pkl
malaria_regressor.pkl
malaria_spike_classifier.pkl
model_evaluation_metrics.csv
model_metadata.json
model_metrics.csv
test_predictions.csv
typhoid_regressor.pkl
typhoid_spike_classifier.pkl


In [24]:
print("Dengue threshold:", spike_thresholds["dengue"])

Dengue threshold: 0.21000000000000002


In [25]:

# Check the available metric columns
print("Available columns:", metrics_df.columns.tolist())

# Separate regression and spike-classification results
task_names = metrics_df["task"].astype(str).str.lower()
spike_metrics = metrics_df[
    task_names.str.contains("spike|class", regex=True)
].copy()

regression_metrics = metrics_df[
    ~task_names.str.contains("spike|class", regex=True)
].copy()

# Regression ranking: lower MAE is better
if not regression_metrics.empty and "MAE" in regression_metrics.columns:
    regression_metrics = regression_metrics.sort_values("MAE")
    print("\nREGRESSION MODELS — lower MAE is better")
    display(regression_metrics)

# Classification ranking: higher F1 is better
f1_col = next(
    (c for c in spike_metrics.columns if c.lower() == "f1"),
    None
)

if not spike_metrics.empty and f1_col:
    spike_metrics = spike_metrics.sort_values(
        f1_col, ascending=False
    )
    print("\nSPIKE CLASSIFIERS — higher F1 is better")
    display(spike_metrics)


Available columns: ['disease', 'task', 'train_rows', 'test_rows', 'MAE', 'RMSE', 'R2', 'baseline_MAE', 'baseline_improvement_pct', 'precision', 'recall', 'F1', 'average_precision', 'threshold', 'validation_F2', 'true_negatives', 'false_positives', 'false_negatives', 'true_positives', 'test_prevalence']

REGRESSION MODELS — lower MAE is better


,disease,task,train_rows,test_rows,MAE,RMSE,R2,baseline_MAE,baseline_improvement_pct,precision,recall,F1,average_precision,threshold,validation_F2,true_negatives,false_positives,false_negatives,true_positives,test_prevalence
4,chikungunya,cases_regression,5820,1440,1.783053,2.334189,0.478892,2.413194,26.112354,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,typhoid,cases_regression,5820,1440,2.135375,2.744496,0.458976,2.890972,26.136444,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10,acute_diarrheal_disease,cases_regression,5820,1440,2.655574,3.518714,0.511842,3.620139,26.644409,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,influenza,cases_regression,5820,1440,2.694950,3.493646,0.479032,3.659028,26.347931,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,malaria,cases_regression,5820,1440,2.793423,3.639029,0.544038,3.733333,25.176178,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,dengue,cases_regression,5820,1440,3.153762,4.165032,0.671725,4.160417,24.195999,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



SPIKE CLASSIFIERS — higher F1 is better


,disease,task,train_rows,test_rows,MAE,RMSE,R2,baseline_MAE,baseline_improvement_pct,precision,recall,F1,average_precision,threshold,validation_F2,true_negatives,false_positives,false_negatives,true_positives,test_prevalence
1,dengue,spike_classification,5740,1440,NaN,NaN,NaN,NaN,NaN,0.274834,0.588652,0.374718,0.328915,0.21,0.532258,1080.0,219.0,58.0,83.0,0.097917
3,malaria,spike_classification,5740,1440,NaN,NaN,NaN,NaN,NaN,0.310345,0.403846,0.350975,0.300852,0.26,0.525060,1144.0,140.0,93.0,63.0,0.108333
7,influenza,spike_classification,5740,1440,NaN,NaN,NaN,NaN,NaN,0.218710,0.865000,0.349142,0.316090,0.14,0.540759,622.0,618.0,27.0,173.0,0.138889
11,acute_diarrheal_disease,spike_classification,5740,1440,NaN,NaN,NaN,NaN,NaN,0.233129,0.655172,0.343891,0.341246,0.18,0.515672,891.0,375.0,60.0,114.0,0.120833
9,typhoid,spike_classification,5740,1440,NaN,NaN,NaN,NaN,NaN,0.178058,0.697183,0.283668,0.249978,0.14,0.525394,841.0,457.0,43.0,99.0,0.098611
5,chikungunya,spike_classification,5740,1440,NaN,NaN,NaN,NaN,NaN,0.147692,0.585366,0.235872,0.204510,0.14,0.484728,1081.0,277.0,34.0,48.0,0.056944


In [26]:
import joblib

model = joblib.load("dengue_regressor.pkl")

print(type(model))
print("Model loaded successfully!")

FileNotFoundError: [Errno 2] No such file or directory: 'dengue_regressor.pkl'

In [27]:
import os

print("Current folder:", os.getcwd())
print("\nFiles in current folder:")
print(os.listdir())

Current folder: C:\Users\YASH\Prime Classes\health2

Files in current folder:
['.ipynb_checkpoints', 'health2.ipynb', 'pune_multi_disease_weekly_synthetic_7years(1).csv', 'trained_models']


In [28]:
import os
import joblib

model_path = os.path.join(
    os.getcwd(),
    "trained_models",
    "dengue_regressor.pkl"
)

print("Model path:", model_path)
print("File exists:", os.path.exists(model_path))

model = joblib.load(model_path)

print(type(model))
print("Model loaded successfully!")

Model path: C:\Users\YASH\Prime Classes\health2\trained_models\dengue_regressor.pkl
File exists: True
<class 'sklearn.pipeline.Pipeline'>
Model loaded successfully!


In [29]:
import os
import joblib

model_dir = os.path.join(os.getcwd(), "trained_models")

models = {}

for filename in os.listdir(model_dir):
    if filename.endswith(".pkl"):
        model_name = filename.replace(".pkl", "")
        models[model_name] = joblib.load(
            os.path.join(model_dir, filename)
        )

print("Total models loaded:", len(models))
print(list(models.keys()))

Total models loaded: 12
['acute_diarrheal_disease_regressor', 'acute_diarrheal_disease_spike_classifier', 'chikungunya_regressor', 'chikungunya_spike_classifier', 'dengue_regressor', 'dengue_spike_classifier', 'influenza_regressor', 'influenza_spike_classifier', 'malaria_regressor', 'malaria_spike_classifier', 'typhoid_regressor', 'typhoid_spike_classifier']
